# Amazon RDS -- Full Load With Historical Data

Gabriel Ferreira

In [1]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import psycopg2
import pandas as pd
import os
from dotenv import load_dotenv

In [2]:
# Star SparkSession
spark = SparkSession.builder \
    .appName("FraudDetectionLakehouse") \
    .config(
        "spark.jars",
        "/home/jovyan/work/jars/hadoop-aws-3.3.4.jar,"
        "/home/jovyan/work/jars/aws-java-sdk-bundle-1.12.262.jar,"
        "/home/jovyan/work/jars/postgresql-42.7.3.jar"
    ) \
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    ) \
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    ) \
    .getOrCreate()

print(spark)

In [3]:
# Configuring AWS credentials

load_dotenv("/home/jovyan/work/.env")

AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

spark._jsc.hadoopConfiguration().set(
    "fs.s3a.access.key",
    AWS_ACCESS_KEY
)

spark._jsc.hadoopConfiguration().set(
    "fs.s3a.secret.key",
    AWS_SECRET_KEY
)

spark._jsc.hadoopConfiguration().set(
    "fs.s3a.endpoint",
    "s3.amazonaws.com"
)

In [4]:
# Read Gold Layer
df_gold = spark.read.parquet(
    "s3a://fraud-detection-data-lake-200702211381/gold/fraud_features/"
)

print(df_gold.count())

1316675


In [5]:
# Checking Current Schema
df_gold.printSchema()

root
 |-- _airbyte_raw_id: string (nullable = true)
 |-- _airbyte_extracted_at: timestamp (nullable = true)
 |-- _airbyte_meta: struct (nullable = true)
 |    |-- sync_id: long (nullable = true)
 |    |-- changes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- field: string (nullable = true)
 |    |    |    |-- change: string (nullable = true)
 |    |    |    |-- reason: string (nullable = true)
 |-- _airbyte_generation_id: long (nullable = true)
 |-- id: long (nullable = true)
 |-- amt: double (nullable = true)
 |-- dob: date (nullable = true)
 |-- job: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- zip: string (nullable = true)
 |-- city: string (nullable = true)
 |-- long: double (nullable = true)
 |-- state: string (nullable = true)
 |-- cc_num: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- category: string (nullable = true)
 |-- city_pop: long (nullable = t

In [6]:
# Dropping column with type struct to avoid errors
df_gold_clean = df_gold.drop("_airbyte_meta")

In [7]:
# Validate clean schema
df_gold_clean.printSchema()

root
 |-- _airbyte_raw_id: string (nullable = true)
 |-- _airbyte_extracted_at: timestamp (nullable = true)
 |-- _airbyte_generation_id: long (nullable = true)
 |-- id: long (nullable = true)
 |-- amt: double (nullable = true)
 |-- dob: date (nullable = true)
 |-- job: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- zip: string (nullable = true)
 |-- city: string (nullable = true)
 |-- long: double (nullable = true)
 |-- state: string (nullable = true)
 |-- cc_num: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- category: string (nullable = true)
 |-- city_pop: long (nullable = true)
 |-- is_fraud: long (nullable = true)
 |-- merchant: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- unix_time: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- ingesti

In [8]:
# JDBC Configuration
jdbc_url = "jdbc:postgresql://fraud-detection-rds.cilk4oyi2c5j.us-east-1.rds.amazonaws.com:5432/fraud_analytics"

properties = {
    "user": "postgres",
    "password": "fraud-gabriel-rds-2026",
    "driver": "org.postgresql.Driver"
}

In [9]:
# Full Load on RDS
df_gold_clean.write.jdbc(
    url=jdbc_url,
    table="fraud_transactions_full",
    mode="overwrite",
    properties=properties
)

print("Full load completed successfully!")

Full load completed successfully!
